# Ejercicio 3: Modelo Vectorial y TF-IDF

## Objetivo de la práctica

- Comprender el modelo vectorial como base para representar documentos y consultas.
- Calcular la matriz TF-IDF para el corpus `data/01_corpus_turismo_500.txt`
- Calcular la matriz TF-IDF para el corpus `Gutenberg 1000`

Nombre: Pérez Pineda Andrés Alejandro

In [22]:
import os
import time
import requests
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import re
import nltk
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
from tqdm import tqdm

# Descargamos los modelos de NLTK (solo se ejecuta realmente la primera vez)
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
stop_words_ingles = set(stopwords.words('english'))

[nltk_data] Downloading package punkt to /home/jovyan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/jovyan/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /home/jovyan/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


### Paso 1: Calcular la matriz TF-IDF para el corpus `data/01_corpus_turismo_500.txt`

In [23]:
def paso1_tfidf_turismo(ruta_archivo):
    print("--- PASO 1: TF-IDF Corpus Turismo ---")
    with open(ruta_archivo, 'r', encoding='utf-8') as f:
        lineas = f.readlines()
    
    # Limpieza básica
    corpus_turismo = [re.sub(r'\\s*', '', linea).strip() for linea in lineas if linea.strip()]
    
    vectorizer_turismo = TfidfVectorizer()
    matriz_tfidf_turismo = vectorizer_turismo.fit_transform(corpus_turismo)
    
    print(f"Corpus Turismo procesado: {len(corpus_turismo)} documentos.")
    print(f"Dimensiones de la matriz TF-IDF: {matriz_tfidf_turismo.shape}")
    print(f"Vocabulario extraído: {len(vectorizer_turismo.get_feature_names_out())} términos.\n")
    
    return vectorizer_turismo, matriz_tfidf_turismo

### Paso 2: Construir el corpus `Gutenberg 1000`

El corpus `Gutenberg 1000` es un corpus compuesto por 1000 libros de Gutenberg Project

In [24]:
def paso2_construir_gutenberg(num_libros=1000, carpeta_destino="data/gutenberg_1000"):
    print("--- PASO 2: Construyendo Corpus Gutenberg ---")
    if not os.path.exists(carpeta_destino):
        os.makedirs(carpeta_destino)
    
    libros_descargados = len(os.listdir(carpeta_destino))
    if libros_descargados >= num_libros:
        print(f"Ya existen {libros_descargados} libros en {carpeta_destino}. Se omite la descarga.\n")
        return
    
    print(f"Descargando {num_libros} libros (esto tomará algo de tiempo)...")
    
    libros_exitosos = 0
    id_libro = 1
    
    while libros_exitosos < num_libros:
        ruta_guardado = os.path.join(carpeta_destino, f"libro_{id_libro}.txt")
        
        if os.path.exists(ruta_guardado):
            libros_exitosos += 1
            id_libro += 1
            continue
            
        url = f"https://www.gutenberg.org/cache/epub/{id_libro}/pg{id_libro}.txt"
        try:
            respuesta = requests.get(url, timeout=5)
            if respuesta.status_code == 200 and "text/plain" in respuesta.headers.get("Content-Type", ""):
                with open(ruta_guardado, 'w', encoding='utf-8') as f:
                    f.write(respuesta.text)
                libros_exitosos += 1
                if libros_exitosos % 50 == 0:
                    print(f"Descargados {libros_exitosos}/{num_libros} libros...")
            time.sleep(0.5) 
        except requests.RequestException:
            pass 
            
        id_libro += 1
    print("Descarga del corpus Gutenberg completada.\n")

# Define las herramientas de preprocesamiento
stemmer = PorterStemmer()

def tokenizador_porter(texto):
    tokens = nltk.word_tokenize(texto)
    # Filtramos: no stopword, es alfabético, y aplicamos stem
    return [stemmer.stem(token) for token in tokens if token.lower() not in stop_words_ingles and token.isalpha()]

def limpiar_gutenberg(texto):
    inicio_marcador = "*** START OF THE PROJECT GUTENBERG"
    fin_marcador = "*** END OF THE PROJECT GUTENBERG"
    
    if inicio_marcador in texto:
        partes = texto.split(inicio_marcador, 1)
        if len(partes) > 1:
            texto = partes[1].split('\n', 1)[-1] 
        
    if fin_marcador in texto:
        texto = texto.split(fin_marcador, 1)[0]
        
    return texto.strip()

### Paso 3: Calcular la matriz TF-IDF para el corpus `Gutenberg 1000`

In [25]:
def paso3_tfidf_gutenberg(carpeta_origen="data/gutenberg_1000"):
    print("--- PASO 3: TF-IDF Corpus Gutenberg ---")
    archivos = [os.path.join(carpeta_origen, f) for f in os.listdir(carpeta_origen) if f.endswith('.txt')]
    
    corpus_gutenberg = []
    nombres_documentos = []
    
    print("Cargando y limpiando libros en memoria (esto tomará varios minutos)...")
    for archivo in tqdm(archivos, desc="Procesando libros", unit="libro"):
        with open(archivo, 'r', encoding='utf-8', errors='ignore') as f:
            texto_limpio = limpiar_gutenberg(f.read())
            corpus_gutenberg.append(texto_limpio)
            nombres_documentos.append(os.path.basename(archivo))
            
    # Vectorizador con tokenizador personalizado
    vectorizer_gutenberg = TfidfVectorizer(
        tokenizer=tokenizador_porter,
        max_features=10000,
        token_pattern=None 
    )
    matriz_tfidf_gutenberg = vectorizer_gutenberg.fit_transform(corpus_gutenberg)
    
    print(f"Corpus Gutenberg procesado: {len(corpus_gutenberg)} documentos.")
    print(f"Dimensiones de la matriz TF-IDF: {matriz_tfidf_gutenberg.shape}\n")
    
    return vectorizer_gutenberg, matriz_tfidf_gutenberg, nombres_documentos

### Paso 4: Programar una función `buscar()` para el corpus `Gutenberg 1000`

In [26]:
def buscar(consulta, vectorizer, matriz_tfidf, nombres_documentos, top_k=5):
    vector_consulta = vectorizer.transform([consulta])
    similitudes = cosine_similarity(vector_consulta, matriz_tfidf).flatten()
    indices_top = similitudes.argsort()[-top_k:][::-1]
    
    print(f"\n--- Resultados de búsqueda para: '{consulta}' ---")
    for i in indices_top:
        similitud = similitudes[i]
        if similitud > 0:
            print(f"Documento: {nombres_documentos[i]} | Similitud: {similitud:.4f}")
        else:
            print("No se encontraron más coincidencias relevantes (Similitud = 0).")
            break

### Ejecución principal

In [27]:
if __name__ == "__main__":
    import os
    
    directorio_base = os.getcwd()
    if os.path.basename(directorio_base) != 'work' and os.path.exists('work'):
        os.chdir('work')
        directorio_base = os.getcwd()
        
    ruta_turismo = os.path.join(directorio_base, "data", "01_corpus_turismo_500.txt")
    
    if not os.path.exists(ruta_turismo):
        print(f"Error: No se encontró el archivo '{ruta_turismo}'.")
    else:
        print("Archivo encontrado. Iniciando proceso...\n")
        
        vec_tur, mat_tur = paso1_tfidf_turismo(ruta_turismo)
        paso2_construir_gutenberg(num_libros=1000) 
        vec_gut, mat_gut, nombres_gut = paso3_tfidf_gutenberg()
        
        consulta_prueba = "science fiction adventure"
        buscar(consulta_prueba, vec_gut, mat_gut, nombres_gut)

Archivo encontrado. Iniciando proceso...

--- PASO 1: TF-IDF Corpus Turismo ---
Corpus Turismo procesado: 500 documentos.
Dimensiones de la matriz TF-IDF: (500, 116)
Vocabulario extraído: 116 términos.

--- PASO 2: Construyendo Corpus Gutenberg ---
Ya existen 1000 libros en data/gutenberg_1000. Se omite la descarga.

--- PASO 3: TF-IDF Corpus Gutenberg ---
Cargando y limpiando libros en memoria (esto tomará varios minutos)...


Procesando libros: 100%|██████████| 1000/1000 [00:06<00:00, 160.91libro/s]


Corpus Gutenberg procesado: 1000 documentos.
Dimensiones de la matriz TF-IDF: (1000, 10000)


--- Resultados de búsqueda para: 'science fiction adventure' ---
Documento: libro_235.txt | Similitud: 0.1740
Documento: libro_728.txt | Similitud: 0.1453
Documento: libro_1016.txt | Similitud: 0.0946
Documento: libro_726.txt | Similitud: 0.0916
Documento: libro_505.txt | Similitud: 0.0802
